# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syed-Attique/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking, produced by a scoring model.** My lane (Lane 2, Refresh / Content Opportunity Scoring) ends in an ordered review queue, not a yes/no. The model gives every page a decline-risk score, and the score is only used to sort pages. Only the top of the list matters, because a reviewer can open a limited number of pages per cycle.

Why not the other task types:
- *Classification* would give a yes/no per page, but the simple "declining" flag already fires on more than half the pages (Section 5), so a yes/no does not tell a reviewer where to start.
- *Clustering* groups similar pages; it does not say which page to open first.

**Action the output supports:** a human reviews the top of the queue and picks one action per page: refresh, rewrite title/meta, expand, merge, or monitor. The score is decision-support. It does not say a refresh will work.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Syed-Attique/FlyRank-Internship"
REPO_DIR = "FlyRank-Internship"

if IN_COLAB:
    # Colab starts with an empty disk, so fetch the repo (the starter CSV lives inside it)
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # Local run: walk up from the notebook folder to the repo root
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found - are you at the repo root?"

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# Same starter filter as the pipeline: some impressions, at least 90 days old, one row per page
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").reset_index(drop=True)
print(f"Lane 2 slice: {len(df):,} pages from {df['client_id'].nunique()} clients")

Lane 2 slice: 30,000 pages from 32 clients


## 2. Target or proxy

**Target (proxy label): `is_declining`**, a page whose search impressions fell by more than the starter threshold between two 30-day windows (`trend_direction == "down"`) **and** had at least 100 impressions in the earlier window (`impressions_prev_30d >= 100`).

- **Where it comes from:** a *defined rule* on observed impressions, not a future outcome. It is a proxy for "this page is really losing visibility". It is built from one current window, so it can include seasonality, a sibling page absorbing demand (consolidation), or noise.
- **Why I added the 100-impression floor:** the lane guide (section 7) says a decline label needs a minimum volume so small wiggles are not counted. I considered the plain starter label (`trend_direction == "down"`) and chose this stricter one. The cost is that my numbers will not be directly comparable to the guide's reported results, which use the plain label.
- **Stronger label for later:** features from the prior 90 days, predicting a decline over the *next* 30 days, which needs the warehouse daily table. The starter CSV has no future window, so I cannot build that here.

The cell below builds both labels so the difference is visible.

In [2]:
down_plain = df["trend_direction"].eq("down")
df["is_declining"] = (down_plain & (df["impressions_prev_30d"] >= 100)).astype(int)

print(f"Plain starter label (trend_direction == 'down'): {down_plain.sum():,} pages ({down_plain.mean():.1%})")
print(f"My label (down AND prev-30d impressions >= 100):  {df['is_declining'].sum():,} pages ({df['is_declining'].mean():.1%})")
dropped = df[down_plain & (df["is_declining"] == 0)]
print(f"Pages dropped by the floor: {len(dropped):,}; their median prev-30d impressions: {dropped['impressions_prev_30d'].median():.0f}")

# What the target column looks like next to the numbers it is built from
df.loc[:, ["content_id", "impressions_prev_30d", "impressions_last_30d", "trend_pct", "trend_direction", "is_declining"]].sample(6, random_state=7)


Plain starter label (trend_direction == 'down'): 16,262 pages (54.2%)
My label (down AND prev-30d impressions >= 100):  11,090 pages (37.0%)
Pages dropped by the floor: 5,172; their median prev-30d impressions: 22


,content_id,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,is_declining
1252,content_a8c35eeef547,27802,26139,-6.0,stable,0
10444,content_26eb8cf5167c,235,19,-91.9,down,1
8994,content_663ee9569ab2,797,652,-18.2,stable,0
7463,content_7e13c4bf098b,0,0,NaN,flat,0
1910,content_b9b75b2367a3,23,7,-69.6,down,0
20606,content_184e5a125c60,3,7,133.3,up,0


## 3. Success metric

**Primary metric: precision@50.** Of the 50 pages the system puts at the top of the queue, how many are `is_declining = 1`? K = 50 is a planning assumption for how many pages a reviewer can look at in a cycle, not a measured fact.

**What "good" means:** a clear gain over two reference points, both measured on **held-out clients** (whole clients kept out of training, so the model is tested on clients it never saw):
1. **Random order.** With a base rate of about 37%, a random top 50 is about 37% right, so a useful model must be clearly above that.
2. **A transparent baseline score,** which I will build in a later week and have to beat.

**Secondary metrics:** average precision (quality of the whole ordering) and recall (how many true declines the queue finds), because missing a real decline is the other kind of mistake. I will also read my top 20 by hand.

The guide reports precision@50 of 0.24 for its baseline rules and 0.74 for a random forest, but on the plain label, so they are a reference for scale only, not a target for my label.

The cell below only checks that the ruler works and shows the random reference.

In [3]:
def precision_at_k(score, y, k=50):
    """Share of the k highest-scored pages that are truly declining."""
    top = np.argsort(-np.asarray(score), kind="stable")[:k]
    return float(np.asarray(y)[top].mean())

y = df["is_declining"].to_numpy()
rng_scores = [precision_at_k(np.random.default_rng(s).random(len(df)), y) for s in range(200)]
print(f"Base rate of is_declining: {y.mean():.3f}")
print(f"Random-order precision@50 (mean of 200 shuffles): {np.mean(rng_scores):.3f}")

Base rate of is_declining: 0.370
Random-order precision@50 (mean of 200 shuffles): 0.368


## 4. The unit of analysis, as a real dataframe

**One row = one page (`content_id`), described by its signals over the trailing 90 days.** The label `is_declining` is attached to the same row.

Three groups of columns (shown in the next cell):
- **Candidate features:** search and engagement signals, page properties, keyword context.
- **The label** and the columns it is *built from*: `trend_direction`, `trend_pct`, `impressions_prev_30d`, `impressions_last_30d`. These must **never** be features, or the model just reads the answer back.
- **Grouping only:** `content_id` and `client_id`, used to split by client, never as features.

A caution I want to check later: the `_90d` totals cover the same days as the two 30-day windows that define the label, so they may partly overlap with it. The cleaner design is a future window (see Section 2).

In [4]:
label_cols = ["is_declining", "trend_direction", "trend_pct", "impressions_prev_30d", "impressions_last_30d"]
id_cols = ["content_id", "client_id"]
feature_cols = [
    "search_volume", "competition", "cpc", "word_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "ctr", "avg_position", "engagement_rate", "scroll_rate",
]

print(f"One row = one page. Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Candidate features: {len(feature_cols)} | label + label-source columns (never features): {len(label_cols)} | ids: {len(id_cols)}")
df[id_cols + feature_cols[:8] + ["is_declining"]].head(8)


One row = one page. Shape: 30,000 rows x 45 columns
Candidate features: 16 | label + label-source columns (never features): 5 | ids: 2


,content_id,client_id,search_volume,competition,cpc,word_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,is_declining
0,content_304f48230142,client_f369cb89fc,10.0,0.67,2.05,3221.0,187,20,3803,29,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,0.05,2481.0,445,25,15320,7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,0.00,3515.0,141,20,12581,11,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,0.00,NaN,463,22,11751,58,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,0.00,2803.0,263,14,19140,24,1
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,1.05,3080.0,147,20,3970,1,1
6,content_9a34b442b552,client_8722616204,0.0,0.00,0.00,3059.0,90,20,20,0,0
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,0.64,NaN,445,22,1724,1,0


## 5. Why ML beats a fixed rule here

I have not trained a model yet, so this section states the argument and the evidence so far; the test comes in later weeks.

- **The simple flag cannot rank.** The plain "declining" flag fires on 54.2% of pages, and even my stricter label marks 37.0%. A reviewer who can open 50 pages needs an order, and a flag is not an order.
- **One-feature rules are unreliable.** Sorting by a single column and taking the top 50 gives very different results depending on which column and which direction (cell below). A rule author has to guess both. The cell shows 12 such sorts; picking the best one on this same data would be cherry-picking, so I only use them to show the spread.
- **The pattern differs by client.** The share of declining pages varies a lot across clients (cell below), so one fixed threshold is unlikely to transfer, which is why I validate by holding out whole clients.
- **What would show ML is not needed:** if a transparent baseline matches the learned ranking on precision@50 for held-out clients, the simple rule should ship, and that is also a result.

Careful words: these numbers describe the 30,000-page starter slice and a proxy label. They show that a ranking is worth testing, not that a model will win.

In [5]:
# (a) Twelve one-feature sorts: 6 columns x 2 directions, top-50 precision against base rate
rows = []
for col in ["impressions_90d", "days_since_last_update", "content_age_days", "search_volume", "word_count", "sessions_90d"]:
    s = df[col].fillna(0)
    rows.append((col, precision_at_k(s, y), precision_at_k(-s, y)))
sorts = pd.DataFrame(rows, columns=["sorted_by", "high_first_p@50", "low_first_p@50"]).round(2)
print(f"Base rate {y.mean():.2f}; one-feature sorts range {sorts[['high_first_p@50','low_first_p@50']].min().min():.2f} to {sorts[['high_first_p@50','low_first_p@50']].max().max():.2f}")
display(sorts)

# (b) Label rate per client (clients with at least 100 pages)
by_client = df.groupby("client_id").agg(pages=("is_declining", "size"), declining_rate=("is_declining", "mean"))
by_client = by_client[by_client["pages"] >= 100]
print(f"{len(by_client)} clients with >=100 pages; declining rate: min {by_client['declining_rate'].min():.0%}, median {by_client['declining_rate'].median():.0%}, max {by_client['declining_rate'].max():.0%}")

Base rate 0.37; one-feature sorts range 0.00 to 0.72


,sorted_by,high_first_p@50,low_first_p@50
0,impressions_90d,0.42,0.00
1,days_since_last_update,0.08,0.00
2,content_age_days,0.46,0.24
3,search_volume,0.32,0.38
4,word_count,0.72,0.34
5,sessions_90d,0.40,0.12


24 clients with >=100 pages; declining rate: min 0%, median 24%, max 90%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.